### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd
import sys
sys.path.append('./utils')

In [2]:
import concurrent
import io
import logging
import re
import re2

import cairosvg
import kagglehub
import torch
from lxml import etree
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

import svg_constraints 

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('DEVICE', DEVICE)

class SVGSanitizer:
    def __init__(self, constraints, default_svg):
        self.constraints = constraints
        self.default_svg = default_svg
    
    def enforce_constraints(self, svg_string: str) -> str:
        """Enforces constraints on an SVG string, removing disallowed elements
        and attributes.

        Parameters
        ----------
        svg_string : str
            The SVG string to process.

        Returns
        -------
        str
            The processed SVG string, or the default SVG if constraints
            cannot be satisfied.
        """
        logging.info('Sanitizing SVG...')

        try:
            parser = etree.XMLParser(remove_blank_text=True, remove_comments=True)
            root = etree.fromstring(svg_string, parser=parser)
        except etree.ParseError as e:
            logging.error('SVG Parse Error: %s. Returning default SVG.', e)
            return self.default_svg
    
        elements_to_remove = []
        for element in root.iter():
            tag_name = etree.QName(element.tag).localname
    
            # Remove disallowed elements
            if tag_name not in self.constraints.allowed_elements:
                elements_to_remove.append(element)
                continue  # Skip attribute checks for removed elements
    
            # Remove disallowed attributes
            attrs_to_remove = []
            for attr in element.attrib:
                attr_name = etree.QName(attr).localname
                if (
                    attr_name
                    not in self.constraints.allowed_elements[tag_name]
                    and attr_name
                    not in self.constraints.allowed_elements['common']
                ):
                    attrs_to_remove.append(attr)
    
            for attr in attrs_to_remove:
                logging.debug(
                    'Attribute "%s" for element "%s" not allowed. Removing.',
                    attr,
                    tag_name,
                )
                del element.attrib[attr]
    
            # Check and remove invalid href attributes
            for attr, value in element.attrib.items():
                 if etree.QName(attr).localname == 'href' and not value.startswith('#'):
                    logging.debug(
                        'Removing invalid href attribute in element "%s".', tag_name
                    )
                    del element.attrib[attr]

            # Validate path elements to help ensure SVG conversion
            if tag_name == 'path':
                d_attribute = element.get('d')
                if not d_attribute:
                    logging.warning('Path element is missing "d" attribute. Removing path.')
                    elements_to_remove.append(element)
                    continue # Skip further checks for this removed element
                # Use regex to validate 'd' attribute format
                path_regex = re2.compile(
                    r'^'  # Start of string
                    r'(?:'  # Non-capturing group for each command + numbers block
                    r'[MmZzLlHhVvCcSsQqTtAa]'  # Valid SVG path commands (adjusted to exclude extra letters)
                    r'\s*'  # Optional whitespace after command
                    r'(?:'  # Non-capturing group for optional numbers
                    r'-?\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?'  # First number
                    r'(?:[\s,]+-?\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?)*'  # Subsequent numbers with mandatory separator(s)
                    r')?'  # Numbers are optional (e.g. for Z command)
                    r'\s*'  # Optional whitespace after numbers/command block
                    r')+'  # One or more command blocks
                    r'\s*'  # Optional trailing whitespace
                    r'$'  # End of string
                )
                if not path_regex.match(d_attribute):
                    logging.warning(
                        'Path element has malformed "d" attribute format. Removing path.'
                    )
                    elements_to_remove.append(element)
                    continue
                logging.debug('Path element "d" attribute validated (regex check).')
        
        # Remove elements marked for removal
        for element in elements_to_remove:
            if element.getparent() is not None:
                element.getparent().remove(element)
                logging.debug('Removed element: %s', element.tag)

        try:
            cleaned_svg_string = etree.tostring(root, encoding='unicode')
            return cleaned_svg_string
        except ValueError as e:
            logging.error(
                'SVG could not be sanitized to meet constraints: %s', e
            )
            return self.default_svg

class SVGProcessor:
    @staticmethod
    def clean_and_extract_svgs(text, default_svg):
        text = re.sub(r'^.*?(<svg\b)', r'\1', text, flags=re.DOTALL)
        svg_blocks = re.findall(r'<svg\b.*?</svg>', text, re.DOTALL)
    
        if svg_blocks:
            tmp = re.findall(r'<svg\b.*?', svg_blocks[-1], re.DOTALL)
            if len(tmp) > 1:
                tmp2 = svg_blocks[-1].split('<svg')
                return '<svg ' + tmp2[-1]
            else:
                return svg_blocks[-1]
        else:
            if "<svg" in text and "</svg>" not in text:
                text += "</svg>"
                return text
            return default_svg
    
    @staticmethod
    def svg_conversion_check(topic, base_svg_code, default_svg):
        try:
            cairosvg.svg2png(bytestring=base_svg_code.encode('utf-8'), write_to="temp.png")
            return base_svg_code
        except Exception as e:
            print(f"Failed to convert {topic} due to {str(e)}, Returning default SVG.")
            return default_svg


class Model:
    def __init__(self):
        self.model="model"
        
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)
        
    def clean_svg(self, base_svg_code: str, max_new_tokens=1024) -> str:
        base_svg_code = SVGProcessor.clean_and_extract_svgs(base_svg_code, self.default_svg)
        clean_svg_code = self.sanitizer.enforce_constraints(base_svg_code)
        return clean_svg_code


DEVICE cuda


In [3]:
model=Model()

### Response Loader from CSV batch files

In [4]:
# import os
# import pandas as pd

# # Directory containing CSV files
# csv_dir = './batches_async/iter3'

# # Get list of all CSV files in the directory
# csv_files = [f for f in os.listdir(csv_dir) if f.endswith('.csv')]

# # Read and combine them
# df_list = [pd.read_csv(os.path.join(csv_dir, file)) for file in csv_files]
# combined_df = pd.concat(df_list, ignore_index=True)

# df=combined_df.copy()
# print(df.shape)

In [5]:
# import re
# def extract_from_gemini_response(text):
#     try:
#         pattern = r'```xml\n(.*?)\n```'
        
#         # Search for the pattern
#         match = re.search(pattern, text, re.DOTALL)
        
#         # Check if a match was found and extract the SVG code
#         if match:
#             svg_code = match.group(1).strip() # .strip() removes leading/trailing whitespace
#             return svg_code
#         else:
#             print("SVG code block not found in the string.")
#             return 0 
#     except Exception as e:
#         return 0

In [6]:
import re
def extract_from_gpt35_response(text):
    try:
        pattern1 = r'```xml\n(.*?)\n```'
        pattern2 = r'```svg\n(.*?)\n```'
        
        # Search for the pattern
        match = re.search(pattern1, text, re.DOTALL)
        
        # Check if a match was found and extract the SVG code
        if match:
            svg_code = match.group(1).strip() # .strip() removes leading/trailing whitespace
            return svg_code
        else:
            match = re.search(pattern2, text, re.DOTALL)
            if match:
                svg_code = match.group(1).strip() # .strip() removes leading/trailing whitespace
                return svg_code
            else:
                if text.strip().startswith("<svg") and text.strip().endswith("</svg>"):
                    return text
                else:
                    match = re.search(r"<svg.*?</svg>", text, re.DOTALL)
                    if match:
                        return match.group(0)
                    else:
                        print("SVG code block not found in the string.")
                        return 0 
                    
    except Exception as e:
        return 0

In [7]:
df1=pd.read_csv('response_1.csv')
df2=pd.read_csv('response_2.csv')
df3=pd.read_csv('response_34.csv')
df4=pd.read_csv('response_56.csv')

df=pd.concat([df1,df2,df3,df4],axis=0)
df=df[['description','response']]

In [8]:
df.shape

(5976, 2)

In [9]:
from tqdm import tqdm
tqdm.pandas()
df['extracted_svg'] = df['response'].progress_apply(lambda x: extract_from_gpt35_response(x))
print(df[df['extracted_svg']!=0].shape)
print(df.shape)

100%|███████████████████████████████████| 5976/5976 [00:00<00:00, 143563.38it/s]

SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.
SVG code block not found in the string.


In [10]:
from tqdm import tqdm
tqdm.pandas()
df=df[df['extracted_svg']!=0]
df['cleaned_svg'] = df['extracted_svg'].progress_apply(lambda x: model.clean_svg(x))
print(df[df['cleaned_svg']!=0].shape)
print(df.shape)

ERROR:root:SVG Parse Error: Namespace prefix xlink for href on use is not defined, line 16, column 32 (<string>, line 16). Returning default SVG.
ERROR:root:SVG Parse Error: Namespace prefix xlink for href on use is not defined, line 6, column 32 (<string>, line 6). Returning default SVG.
ERROR:root:SVG Parse Error: Namespace prefix xlink for href on use is not defined, line 6, column 57 (<string>, line 6). Returning default SVG.
ERROR:root:SVG Parse Error: AttValue: " or ' expected, line 18, column 24 (<string>, line 18). Returning default SVG.
ERROR:root:SVG Parse Error: Opening and ending tag mismatch: rect line 3 and svg, line 12, column 7 (<string>, line 12). Returning default SVG.
ERROR:root:SVG Parse Error: AttValue: " or ' expected, line 9, column 25 (<string>, line 9). Returning default SVG.
ERROR:root:SVG Parse Error: Namespace prefix xlink for href on use is not defined, line 17, column 32 (<string>, line 17). Returning default SVG.
ERROR:root:SVG Parse Error: Namespace pref

(5940, 4)
(5940, 4)


In [11]:
df['extracted_svg'].iloc[0]

'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 256 256" width="256" height="256">\n  <!-- Background - Golden Wheat Field -->\n  <rect x="0" y="0" width="256" height="256" fill="#D4AC0D" />\n  \n  <!-- Barn -->\n  <rect x="50" y="50" width="150" height="150" fill="#A52A2A" />\n  <rect x="70" y="100" width="30" height="100" fill="#8B4513" />\n  <rect x="130" y="100" width="30" height="100" fill="#8B4513" />\n  <polygon points="50,50 125,20 200,50" fill="#8B4513" />\n  <line x1="50" y1="50" x2="200" y2="50" stroke="#000" stroke-width="2" />\n  \n</svg>'

In [12]:
import sys
sys.path.append('./utils')
from siglip_class import SVGMetricEvaluator
from aesthetic_evaluator import AestheticEvaluator

In [13]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError
import pandas as pd
from tqdm import tqdm
tqdm.pandas()
# Initialize evaluator
evaluator = SVGMetricEvaluator()
aes_eval = AestheticEvaluator()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [14]:
def safe_svg_metric(prompt, svg, timeout=20):
    with ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(evaluator.svg_metric, prompt, svg)
        try:
            return future.result(timeout=timeout)
        except TimeoutError:
            print("Timeout: Skipping slow SVG evaluation.")
            return 0
        except Exception as e:
            print(f"Error: {e}")
            return 0

def safe_aes_eval(svg, timeout=10):
    with ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(aes_eval.get_score, svg)
        try:
            return future.result(timeout=timeout)
        except TimeoutError:
            print("Timeout: Skipping slow SVG evaluation.")
            return 0
        except Exception as e:
            print(f"Error: {e}")
            return 0

In [15]:
# Apply with tqdm and timeout-aware function
#df = df.drop(df.index[3061])
df['cleaned_svg_sl_score'] = df.progress_apply(
    lambda row: safe_svg_metric(row['description'], row['cleaned_svg']), axis=1
)

100%|███████████████████████████████████████| 5940/5940 [06:14<00:00, 15.87it/s]


In [16]:
df[df['cleaned_svg_sl_score'] > 0.5].shape

(452, 5)

In [17]:
df.to_csv('cleaned_svg.csv',index=False)

In [18]:
#safe_svg_metric(df['description'].iloc[3062], df['extracted_svg'].iloc[3062])

### Async version is actually slower, why?

In [19]:
# import asyncio
# import nest_asyncio
# import pandas as pd
# from tqdm import tqdm
# from concurrent.futures import ThreadPoolExecutor

# nest_asyncio.apply()

# # Evaluators
# evaluator = SVGMetricEvaluator()
# aes_eval = AestheticEvaluator()

# # Function to evaluate one row
# def evaluate_row(row):
#     description = row['description']
#     extracted_svg = row['extracted_svg']
#     cleaned_svg = row['cleaned_svg']
    
#     return {
#         'gemini_extracted_svg_sl_score': evaluator.svg_metric(description, extracted_svg),
#         'gemini_cleaned_svg_sl_score': evaluator.svg_metric(description, cleaned_svg),
#         'gemini_extracted_svg_aes_score': aes_eval.get_score(extracted_svg),
#         'gemini_cleaned_svg_aes_score': aes_eval.get_score(cleaned_svg)
#     }

# # Async processor
# async def process_dataframe_async(df, max_workers=8):
#     loop = asyncio.get_event_loop()
#     results = []
    
#     with ThreadPoolExecutor(max_workers=max_workers) as executor:
#         tasks = [
#             loop.run_in_executor(executor, evaluate_row, row)
#             for _, row in df.iterrows()
#         ]
#         for result in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
#             results.append(await result)
    
#     return results

# # Run and merge results
# results = await process_dataframe_async(df)
# results_df = pd.DataFrame(results)
# df = pd.concat([df.reset_index(drop=True), results_df], axis=1)

In [20]:
#df[df['gemini_cleaned_svg_sl_score'] >0.5].to_csv('gemini_filterd.csv',index=False)